In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [43]:
df=pd.read_csv("dataset/DataCoSupplyChainDataset.csv",encoding='latin1')

In [44]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [46]:
df.isna().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


Since both Order ZipCode and Product Description Have more then 80% Null values we will drop both the columns

In [47]:
df.drop(columns=['Order Zipcode','Product Description'],inplace=True)

In [48]:
df['Customer Lname']=df['Customer Lname'].fillna("Unknown")

In [49]:
# dropping the 3 rows which have null as zip 
df.dropna(subset=["Customer Zipcode"],inplace=True)

In [50]:
# checking again
df.isna().sum().sum()

np.int64(0)

In [51]:
sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.facecolor'] = '#00000000'

In [52]:
fig=px.histogram(df,x="Type",color="Delivery Status")
fig.update_layout(bargap=0.1)
fig.show()

## Payment Type & Delivery Behavior

DEBIT is the dominant payment method, followed by TRANSFER CASH is rarely used,
suggesting this is primarily a B2B or digitally-native customer base.

More importantly, Late Delivery appears as the most frequent delivery status across
ALL payment types. This tells us delays are not payment method specific  they are
a systemic logistics problem, which makes them predictable with the right features.The most common method of payment is DEBIT followed by TRANSFER . CASH seems to be the least preffered mode. Late Delivery is the most oftern recurring category

In [53]:
fig=px.histogram(df,x="Shipping Mode",color="Delivery Status")
fig.update_layout(bargap=0.1)
fig.show()


## Shipping Mode Volume vs Delay

Standard Class handles the largest order volume by far. However, volume alone is
misleading  Standard Class appears to have a more balanced split between on-time
and late deliveries, while First Class and Second Class are dominated by late delivery.

This raises a key question: are premium shipping modes actually failing their
customers more often? The next chart answers this with rates, not raw counts.

In [54]:
# Late delivery rate by Order Region
region_delay = df.groupby('Order Region')['Late_delivery_risk'].mean().sort_values(ascending=False).reset_index()
region_delay.columns = ['Region', 'Late Rate']
region_delay['Late Rate %'] = (region_delay['Late Rate'] * 100).round(1)

fig = px.bar(region_delay, x='Region', y='Late Rate %', 
             title='Late Delivery Rate by Region',
             color='Late Rate %', color_continuous_scale='Reds')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Geographic Delay Patterns

Central Africa has the highest late delivery rate at 58%, followed by South Asia.
Canada sits at the lowest end (~48%). However, the striking insight here is how
NARROW this range is  all regions fall between 48% and 58%.

This means geography alone is not a strong discriminator. The real delay drivers
must lie elsewhere  likely in shipping mode choice and order processing time.
These regional features will still be included in the model but are not expected
to be top predictors.

In [55]:
df['order date (DateOrders)']=pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)']=pd.to_datetime(df['shipping date (DateOrders)'])

df['delay_gap']=df['Days for shipping (real)']-df['Days for shipment (scheduled)']

fig=px.histogram(df,x='delay_gap', color='Late_delivery_risk',
                   title='Distribution of Delay Gap (Real - Scheduled Days)',
                   nbins=30,
                   color_discrete_map={0: '#2ecc71', 1: '#e74c3c'})
fig.show()

## Delay Gap Distribution — The Core Signal

delay_gap = Days for shipping (real) — Days for shipment (scheduled)

Positive values mean the shipment took longer than promised. Negative means it
arrived early. The distribution clearly separates into two clusters by Late_delivery_risk:

- Risk = 0 (green): tightly clustered at 0 and negative values — delivered on time or early
- Risk = 1 (red): concentrated at +1 to +3 days — consistently overshooting the schedule

This confirms that Late_delivery_risk is a reliable and clean target variable.
The delay_gap column itself will likely be our most powerful engineered feature in modelling.

In [56]:
shipping_delay=df.groupby('Shipping Mode')["Late_delivery_risk"].mean().sort_values(ascending=False).reset_index()
shipping_delay['Late rate %']=(shipping_delay["Late_delivery_risk"]*100).round(1)

fig=px.bar(shipping_delay,y='Late rate %',x="Shipping Mode",color="Late rate %",color_continuous_scale="Oranges",title="Late Delivery Rate by Shipping Mode",text="Late rate %")
fig.show()

## Shipping Mode  Late Delivery Rate (%)

First Class: 95.3% late rate. Second Class: 76.4%. Standard Class: 38%.

This is the most striking finding in the entire EDA. Premium shipping modes
have dramatically HIGHER late delivery rates  the opposite of what customers expect.

The likely explanation: First Class and Second Class promise faster delivery windows
(1–2 days), which are extremely difficult to honour consistently. Standard Class
sets conservative 4–5 day expectations, making it far easier to deliver on time.

Business implication: Customers paying more for speed are the most likely to be
disappointed. Shipping Mode will almost certainly be a top feature in our classifier.

In [57]:
dept_delay=df.groupby("Department Name")['Late_delivery_risk'].mean().sort_values(ascending=False).reset_index()
dept_delay['Late rate %']=(dept_delay['Late_delivery_risk']*100).round(1)

fig=px.bar(dept_delay,x='Department Name',y='Late rate %',text="Late rate %",color="Late rate %",color_continuous_scale=["white", "pink", "hotpink"],title="Late Delivery Rate by Depratments")
fig.show()

## Late Delivery Rate by Department

Pet Shop has the highest late delivery rate, but the range across all departments
is remarkably tight  between 55% and 58%.

This near-uniform distribution tells us something important: delays are NOT
product or department specific. No single department is being prioritised or
neglected in fulfillment.

For feature engineering, this means Department Name will likely have low predictive
power on its own. We will include it but not engineer heavy features around it.
The real signal lies in shipping mode, geography, and order processing time.

In [58]:
df['order_month']=df['order date (DateOrders)'].dt.month
month_delay=df.groupby("order_month")['Late_delivery_risk'].mean().reset_index()
month_delay['Late rate %']=(month_delay['Late_delivery_risk']*100).round(1)
month_delay['Month Name']=pd.to_datetime(month_delay.order_month,format="%m").dt.strftime('%b')
fig = px.line(month_delay, x='Month Name', y='Late rate %',
              title='Late Delivery Rate by Month',
              markers=True)
fig.show()


## Seasonal Delay Patterns

August shows the highest late delivery spike in the dataset. Contrary to common
assumption, Q4 (October–December) does NOT show a significant holiday driven surge
in this data.

This suggests delays in this supply chain are driven more by operational factors
(shipping mode, processing time) than by seasonal demand spikes. The August spike
may be worth investigating further it could reflect a supplier-side issue, a
regional weather pattern, or a data artefact.

For feature engineering: we will still create a holiday_window flag for Nov–Dec
as a precaution, but it may not end up as a top predictor.

In [59]:

balance = df['Late_delivery_risk'].value_counts(normalize=True).sort_index() * 100
print(f"On time: {balance[0]:.1f}%")
print(f"Late: {balance[1]:.1f}%")

fig = px.pie(values=balance.values, names=['On Time', 'Late'],
             title='Target Variable Distribution',
             color_discrete_sequence=[ '#e74c3c','#2ecc71'])
fig.show()

On time: 45.2%
Late: 54.8%


## Target Variable Distribution Class Balance Check

On Time: 45.2% | Late: 54.8% 

The split is near-balanced. No special handling needed for class imbalance.
Standard training will work fine.

In [60]:
# Droping PII and irrelevant columns before feature engineering
cols_to_drop = ['Customer Email', 'Customer Password', 'Customer Fname', 
                'Customer Lname', 'Customer Street', 'Customer Zipcode',
                'Product Image', 'Delivery Status']

df.drop(columns=cols_to_drop, inplace=True)
print(f"Remaining columns: {df.shape[1]}")

Remaining columns: 45


## Columns Dropped Before Feature Engineering

Removed 8 columns for the following reasons:

- Customer Email, Password, Fname, Lname, Street, Zipcode → Personal Identifiable
  Information (PII). No predictive value and should never enter a model.

- Product Image → URL string. Not useful for tabular ML.

- Delivery Status → CRITICAL: this column directly encodes the target variable
  (Late/On Time/Advance). Keeping it would cause data leakage  the model would
  learn from the answer itself. Dropped before any modelling step.

Remaining columns: 45  ← this prints automatically from your code

In [61]:
df=df.sort_values("order date (DateOrders)")

 Since the Date is a time series Date if we randomely split the data , data bleeding might occur hence we are ordering the data on Order date before splitting it

In [62]:
import os
os.makedirs('dataset/processed', exist_ok=True)
df.to_csv('dataset/processed/clean_data.csv', index=False)
print("Saved.")

Saved.
